# MinCED CRISPR Array Detection

![MinCED CRISPR Array Detection](https://proto-bio.github.io/proto-assets/images/tool/minced/hero.png)

This notebook demonstrates `run_minced`, which scans nucleotide sequences for the characteristic repeat-spacer architecture of CRISPR arrays. Given raw genomic DNA, MinCED reports each detected array with its repeat consensus, individual spacers, and genomic coordinates. CRISPR detection is typically the first step when characterizing a CRISPR system in a newly sequenced microbe or identifying candidate loci for a Cas9 engineering pipeline.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("minced")
display_overview("minced")
display_docs_section("minced", "Background")

# MinCED

[MinCED](https://github.com/ctSkennerton/minced) (Mining CRISPRs in Environmental Datasets) is a fast Java program that locates [CRISPR](https://en.wikipedia.org/wiki/CRISPR) arrays in nucleotide sequences from isolate genomes or metagenomic contigs. It returns each detected array as an ordered list of repeat-spacer units with their genomic coordinates, the repeat consensus, and per-spacer sequence.

MinCED is a derivative of the [CRISPR Recognition Tool (CRT)](https://pmc.ncbi.nlm.nih.gov/articles/PMC1924867/) ([Bland et al., 2007](https://doi.org/10.1186/1471-2105-8-209)), maintained by [Connor Skennerton](https://github.com/ctSkennerton). CRISPR arrays are blocks of short, near-identical direct repeats (typically 23 to 47 nt) separated by unique [spacer](https://en.wikipedia.org/wiki/CRISPR#Spacer_acquisition) sequences (typically 26 to 50 nt) that record fragments of past viral and plasmid infections; they form the heritable memory of the [CRISPR-Cas adaptive immune system](https://en.wikipedia.org/wiki/CRISPR) of bacteria and archaea.

Internally, MinCED uses a [k-mer](https://en.wikipedia.org/wiki/K-mer) seed-and-extend strategy. It scans for short exact k-mer matches that recur at a consistent spacing, then extends each seed bidirectionally to the actual repeat length, and finally validates the candidate by checking that the inter-repeat spacers fall within the configured length window. The algorithm runs on raw DNA, has linear time complexity in sequence length, and finishes in seconds on a typical 5 Mb bacterial genome on commodity CPU hardware.

## Available tools

In [2]:
display_available_tools("minced")

- **`run_minced()`** — Detect CRISPR arrays in nucleotide sequences using MinCED

### `run_minced`

`run_minced` detects CRISPR arrays in nucleotide sequences using MinCED (Mining CRISPRs in Environmental Datasets). The tool scans for the repeat-spacer structure characteristic of CRISPR loci and returns each array with its repeat sequences, spacer sequences, and positions. The `min_num_repeats` and `min_repeat_length` thresholds control detection sensitivity.

In [3]:
from proto_tools import (
    MincedInput,
    MincedConfig,
    run_minced,
)

In [4]:
# Display input docs
display_api_reference("minced", "input", "run_minced")

from pathlib import Path

# Real Streptococcus pyogenes SF370 CRISPR1 array region (NC_002737.2).
crispr_sequence = "".join(l for l in Path("spyogenes_crispr_locus.fasta").read_text().splitlines() if not l.startswith(">"))
inputs = MincedInput(sequences=[crispr_sequence])

**Input** — `MincedInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Nucleotide sequence(s) to search for CRISPR arrays |

In [5]:
# Display config docs
display_api_reference("minced", "config", "run_minced")

# Require at least 3 repeats of at least 27nt | see docs above for all fields
config = MincedConfig(
    min_num_repeats=3,
    min_repeat_length=27,
)

**Config** — `MincedConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>min_num_repeats</code> | <code>int</code> | <code>3</code> | Min repeats per array; raise to 4+ for high-confidence |
| <code>min_repeat_length</code> | <code>int</code> | <code>23</code> | Min repeat length in nt; below 23 risks tandem-repeat hits |
| <code>max_repeat_length</code> | <code>int</code> | <code>47</code> | Max repeat length in nt; covers known CRISPR families, raise for unusual loci |
| <code>min_spacer_length</code> | <code>int</code> | <code>26</code> | Min spacer length in nt; lower (~18) for partial or degraded arrays |
| <code>max_spacer_length</code> | <code>int</code> | <code>50</code> | Max spacer length in nt; raise for noncanonical families with longer spacers |

In [6]:
# Run CRISPR array detection
result = run_minced(inputs, config)

Running run_minced [00:00]

In [7]:
# Display output docs
display_api_reference("minced", "output", "run_minced")

# Report detected arrays, repeat counts, and the first few spacers per array
for seq_result in result.results:
    print(f"{seq_result.sequence_id}: {seq_result.num_arrays} CRISPR array(s) found")
    if seq_result.has_crispr:
        for i, array in enumerate(seq_result.crispr_arrays):
            print(f"  Array {i+1}: {array.num_repeats} repeats")
            print(f"  Spacers: {array.spacers[:3]}..." if len(array.spacers) > 3 else f"  Spacers: {array.spacers}")

**Output** — `MincedOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[MincedSequenceResult]</code> | <code>[]</code> | Per-sequence CRISPR array detection results |

**`MincedSequenceResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_id</code> | <code>str</code> | required | ID of the input sequence |
| <code>crispr_arrays</code> | <code>list[CrisprArray]</code> | <code>[]</code> | CRISPR arrays detected in this sequence |

**`CrisprArray`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>repeats_and_spacers</code> | <code>list[CrisprRepeatSpacer]</code> | <code>[]</code> | List of repeat-spacer units in this CRISPR array |

**`CrisprRepeatSpacer`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>position</code> | <code>int</code> | required | 1-indexed start position of the repeat on the input sequence |
| <code>repeat</code> | <code>str</code> | required | Repeat sequence |
| <code>spacer</code> | <code>str &#124; None</code> | <code>None</code> | Spacer sequence (None for last repeat) |
| <code>repeat_length</code> | <code>int &#124; None</code> | <code>None</code> | Length of the repeat |
| <code>spacer_length</code> | <code>int &#124; None</code> | <code>None</code> | Length of the spacer |

seq_0: 1 CRISPR array(s) found
  Array 1: 7 repeats
  Spacers: ['TGCGCTGGTTGATTTCTTCTTGCGCTTTTT', 'TTATATGAACATAACTCAATTTGTAAAAAA', 'AGGAATATCCGCAATAATTAATTGCGCTCT']...


## Export Results

CRISPR detection results can be exported to CSV or JSON. CSV produces one row per repeat-spacer unit, making it easy to filter or aggregate in spreadsheet tools. JSON preserves the full nested array structure for programmatic consumption.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="minced_results", export_path=output_dir, file_format="json")
print(f"CRISPR detection results exported to {output_dir}")

CRISPR detection results exported to example_output
